# Interval Multiplicities Computations (large grid bifiltration)

This notebook computes the interval multiplicity invariant for a finite-grid persistence module `M` after Scheme 1 has produced its minimal projective resolution and minimal injective coresolution.

Step 1 filters intervals `I` whose interval multiplicity `d_M(V_I)` may be nonzero. Later sections will compute concrete invariants on the retained candidates.


## Step 1: Interval Candidate Filtering

This section runs the conservative filtering step before any multiplicity formula is evaluated. It only produces candidate intervals for later downstream invariant computations.

The input files should be the Scheme 1 outputs from `resolution_compt_test.ipynb`:

- `<prefix>.finite_grid.projective_resolution.txt`
- `<prefix>.finite_grid.injective_coresolution.txt`

The finite grid is the compressed grid `[0,n] x [0,m]`, so the grid minimum is always `(0, 0)`.


In [ ]:
from pathlib import Path

from finite_grid_interval_candidates import (
    CandidateGenerationLimitError,
    compute_interval_candidates,
    compute_radical_layer_supports_from_projective_txt,
    compute_socle_layer_supports_from_injective_txt,
    filter_interval_candidates,
    load_interval_candidates_text,
    load_resolution_support_data,
)
from intmult_from_filtration import (
    compare_interval_multiplicity_results,
    compute_interval_multiplicities_from_candidate_result,
    compute_interval_multiplicities_from_injective_candidate_result,
    load_interval_multiplicities_text,
)
from interval_multiplicity_visualization import (
    interval_multiplicity_visualization_mode,
    load_grid_compression_from_scc2020,
    plot_connected_persistence_diagram_binned_histogram_for_two_row_filtration,
    plot_connected_persistence_diagram_for_two_row_filtration,
    plot_interval_multiplicity_binned_2d_histogram,
    plot_interval_multiplicities_auto,
    plot_interval_multiplicities_on_grid,
    plot_interval_multiplicity_overlay,
    summarize_interval_multiplicity_result,
)
from utils import Interval

PROJECT_ROOT = Path.cwd()
PROJECT_ROOT

## Configure Paths

Set `INPUT_PREFIX` to the Scheme 1 output prefix. The default uses the small `3x3_bifiltration` example.

In [ ]:
RESOLUTION_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "resolution_computation"
CHAIN_COMPLEX_DIR = PROJECT_ROOT / "data" / "chain_complex_scc2020"
CANDIDATE_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "interval_candidates"
MULTIPLICITY_OUTPUT_DIR = PROJECT_ROOT / "outputs" / "interval_multiplicities"

# INPUT_PREFIX = "3x3_bifiltration"
INPUT_PREFIX = "two_circles_200pts_knn_function.alpha.scc2020"
# INPUT_PREFIX = "chromatic_four_circles_500pts_function.alpha.scc2020"

OUTPUT_PREFIX = INPUT_PREFIX[:-len(".scc2020")] if INPUT_PREFIX.endswith(".scc2020") else INPUT_PREFIX

CHAIN_COMPLEX_TXT = CHAIN_COMPLEX_DIR / f"{INPUT_PREFIX}.txt"
PROJECTIVE_TXT = RESOLUTION_OUTPUT_DIR / f"{INPUT_PREFIX}.finite_grid.projective_resolution.txt"
INJECTIVE_TXT = RESOLUTION_OUTPUT_DIR / f"{INPUT_PREFIX}.finite_grid.injective_coresolution.txt"
CANDIDATE_TXT = CANDIDATE_OUTPUT_DIR / f"{INPUT_PREFIX}.interval_candidates.txt"
MULTIPLICITY_TXT = MULTIPLICITY_OUTPUT_DIR / f"{OUTPUT_PREFIX}.interval_multiplicities.projective.txt"
INJECTIVE_MULTIPLICITY_TXT = MULTIPLICITY_OUTPUT_DIR / f"{OUTPUT_PREFIX}.interval_multiplicities.injective.txt"

print("Original chain complex:", CHAIN_COMPLEX_TXT)
print("Original chain complex exists:", CHAIN_COMPLEX_TXT.exists())
print("Projective resolution:", PROJECTIVE_TXT)
print("Projective exists:", PROJECTIVE_TXT.exists())
print("Injective coresolution:", INJECTIVE_TXT)
print("Injective exists:", INJECTIVE_TXT.exists())
print("Candidate output:", CANDIDATE_TXT)

## Load Critical-Set Supports

The filter starts from four critical supports:

- `P_0` grades as `supp(top M)`
- `P_1` grades as `supp(top P_1(M))`
- `Q^0` grades as `supp(soc M)`
- `Q^1` grades as `supp(soc Q^1(M))`

This cell also computes higher socle layer supports from the minimal injective copresentation matrix `Q^0 -> Q^1` using the Section 3.1 local rank formula. Set `SOCLE_LAYER_MAX` in the next cell to control the largest layer `r`; the allowed maximum is `n + m + 1` for grid max `(n, m)`.

In [ ]:
support_data = load_resolution_support_data(PROJECTIVE_TXT, INJECTIVE_TXT)
# SOCLE_LAYER_MAX = min(200, support_data.bounds.max_grade[0] + support_data.bounds.max_grade[1] + 1)
# socle_layer_result = compute_socle_layer_supports_from_injective_txt(
#     INJECTIVE_TXT,
#     max_layer=SOCLE_LAYER_MAX,
#     min_layer=2,
# )

print("Grid max:", support_data.bounds.max_grade)
print("supp(top M) from P_0:", support_data.top_p0_support)
print("supp(top P_1(M)) from P_1:", support_data.top_p1_support)
print("supp(soc M) from Q^0:", support_data.soc_q0_support)
print("supp(soc Q^1(M)) from Q^1:", support_data.soc_q1_support)
# print(f"socle layer supports for r=2..{SOCLE_LAYER_MAX}:")
# for layer in socle_layer_result.layers:
#     print(f"supp(soc^{layer.layer} M / soc^{layer.layer - 1} M):", layer.support)

## Run Candidate Filtering

This step uses cheap necessary checks before computing proper source/sink data.  `USE_SOCLE_LAYER_FILTER` applies the interval-socle-layer necessary condition for an embedding `V_I -> M`; `USE_RADICAL_LAYER_FILTER` applies the radical-layer necessary condition for an epimorphism `M -> V_I`.  `LOAD_EXISTING_CANDIDATES` is available for small files, but large candidate txt files may be slower to reconstruct than to recompute.

In [ ]:
CRITERION = "crt1_2d"
GENERATION_MODE = "support_antichains"
MAX_ANTICHAINS = None
MAX_CANDIDATES = None
LOAD_EXISTING_CANDIDATES = True  # Loading very large candidate txt files can be slower than optimized recomputation.

# Optional necessary filter for interval summands: since an interval summand
# embeds into M, its interval socle layers must be contained in the corresponding
# socle layers of M.  This can reduce candidate counts on larger examples.
USE_SOCLE_LAYER_FILTER = True
SOCLE_LAYER_FILTER_CAP = None  # None means use n + m + 1 and stop at the first empty layer.
USE_RADICAL_LAYER_FILTER = True
RADICAL_LAYER_FILTER_CAP = None  # None means use n + m + 1 and stop at the first empty layer.

socle_filter_supports = None
socle_filter_r = None
if USE_SOCLE_LAYER_FILTER:
    max_allowed_socle_layer = (
        support_data.bounds.max_grade[0]
        + support_data.bounds.max_grade[1]
        + 1
    )
    effective_socle_cap = (
        max_allowed_socle_layer
        if SOCLE_LAYER_FILTER_CAP is None
        else int(SOCLE_LAYER_FILTER_CAP)
    )
    socle_filter_result = compute_socle_layer_supports_from_injective_txt(
        INJECTIVE_TXT,
        max_layer=effective_socle_cap,
        stop_at_empty=True,
        support_only=True,
    )
    first_empty_socle_layer = next(
        (layer.layer for layer in socle_filter_result.layers if not layer.support),
        None,
    )
    socle_filter_supports = {
        layer.layer: frozenset(layer.support)
        for layer in socle_filter_result.layers
        if first_empty_socle_layer is None or layer.layer < first_empty_socle_layer
    }
    if first_empty_socle_layer is None:
        socle_filter_r = max(socle_filter_supports, default=1)
    else:
        socle_filter_r = first_empty_socle_layer
    print("Socle-layer filter enabled; layer cap =", socle_filter_r)

# Optional necessary filter for interval summands: since V_I is also a quotient
# of M, the radical layers of V_I must be contained in the corresponding
# radical layers of M.  Radical layer j means rad^(j-1) / rad^j, so indices start at 1.
radical_filter_supports = None
radical_filter_r = None
if USE_RADICAL_LAYER_FILTER:
    max_allowed_radical_layer = (
        support_data.bounds.max_grade[0]
        + support_data.bounds.max_grade[1]
        + 1
    )
    effective_radical_cap = (
        max_allowed_radical_layer
        if RADICAL_LAYER_FILTER_CAP is None
        else int(RADICAL_LAYER_FILTER_CAP)
    )
    radical_filter_result = compute_radical_layer_supports_from_projective_txt(
        PROJECTIVE_TXT,
        max_layer=effective_radical_cap,
        stop_at_empty=True,
        support_only=True,
    )
    first_empty_radical_layer = next(
        (layer.layer for layer in radical_filter_result.layers if not layer.support),
        None,
    )
    radical_filter_supports = {
        layer.layer: frozenset(layer.support)
        for layer in radical_filter_result.layers
        if first_empty_radical_layer is None or layer.layer < first_empty_radical_layer
    }
    if first_empty_radical_layer is None:
        radical_filter_r = max(radical_filter_supports, default=1)
    else:
        radical_filter_r = first_empty_radical_layer
    print("Radical-layer filter enabled; layer cap =", radical_filter_r)

if LOAD_EXISTING_CANDIDATES and CANDIDATE_TXT.exists() and not USE_SOCLE_LAYER_FILTER and not USE_RADICAL_LAYER_FILTER:
    candidate_result = load_interval_candidates_text(CANDIDATE_TXT)
    print("Loaded existing candidate txt:", CANDIDATE_TXT)
else:
    try:
        candidate_result = compute_interval_candidates(
            PROJECTIVE_TXT,
            INJECTIVE_TXT,
            output_path=CANDIDATE_TXT,
            criterion=CRITERION,
            generation_mode=GENERATION_MODE,
            max_antichains=MAX_ANTICHAINS,
            max_candidates=MAX_CANDIDATES,
            socle_layer_supports=socle_filter_supports,
            socle_filter_r=socle_filter_r,
            radical_layer_supports=radical_filter_supports,
            radical_filter_r=radical_filter_r,
        )
    except CandidateGenerationLimitError as exc:
        print("Candidate generation stopped:", exc)
        print("Set MAX_ANTICHAINS or MAX_CANDIDATES to None for an exhaustive run.")
        raise

print("Generated source/sink pairs:", candidate_result.generated_count)
print("Retained candidates:", candidate_result.retained_count)
print("Candidate txt:", candidate_result.output_path)


## Step 2: Interval Multiplicities

Use the projective presentation formula to evaluate exact multiplicities only on the retained Step 1 candidates. If Step 1 retained no candidates, this cell skips the rank computation and writes a short report.

In [ ]:
multiplicity_result = compute_interval_multiplicities_from_candidate_result(
    candidate_result,
    output_path=MULTIPLICITY_TXT,
)

print("Skipped:", multiplicity_result.skipped)
print("Candidate intervals:", multiplicity_result.candidate_count)
print("Nonzero interval multiplicities:", len(multiplicity_result.nonzero_multiplicities))
print("Saved multiplicity txt:", multiplicity_result.output_path)
print("Message:", multiplicity_result.message)

for record in multiplicity_result.nonzero_multiplicities[:20]:
    print(
        f"interval {record.interval_index}: multiplicity={record.multiplicity}, "
        f"src={record.candidate.src}, snk={record.candidate.snk}"
    )
if len(multiplicity_result.nonzero_multiplicities) > 20:
    print(f"... {len(multiplicity_result.nonzero_multiplicities) - 20} more nonzero intervals in the txt output")


## Injective Copresentation Double Check

Compute the same interval multiplicities from the injective copresentation matrix and compare candidate-by-candidate with the projective-presentation result.


In [ ]:
injective_multiplicity_result = compute_interval_multiplicities_from_injective_candidate_result(
    candidate_result,
    output_path=INJECTIVE_MULTIPLICITY_TXT,
)
comparison_result = compare_interval_multiplicity_results(
    multiplicity_result,
    injective_multiplicity_result,
    candidates=candidate_result.candidates,
)

print("Injective nonzero interval multiplicities:", len(injective_multiplicity_result.nonzero_multiplicities))
print("Saved injective multiplicity txt:", injective_multiplicity_result.output_path)
print("Methods consistent:", comparison_result.consistent)
print("Mismatches:", len(comparison_result.mismatches))
for interval_index, projective_mult, injective_mult, candidate in comparison_result.mismatches[:20]:
    print(
        f"interval {interval_index}: projective={projective_mult}, "
        f"injective={injective_mult}, src={candidate.src}, snk={candidate.snk}"
    )


## Preview Candidate and Multiplicity Output


In [ ]:
def preview_text_file(path, max_lines=80):
    path = Path(path)
    lines = path.read_text(encoding="utf-8").splitlines()
    for line in lines[:max_lines]:
        print(line)
    if len(lines) > max_lines:
        print(f"... ({len(lines) - max_lines} more lines)")


print("Candidate output preview")
preview_text_file(CANDIDATE_TXT, max_lines=80)

print("\nProjective multiplicity output preview")
preview_text_file(MULTIPLICITY_TXT, max_lines=120)

print("\nInjective multiplicity output preview")
preview_text_file(INJECTIVE_MULTIPLICITY_TXT, max_lines=120)



## Visualize Nonzero Intervals

This cell has independent display switches. For a two-row compressed grid, `DRAW_CONNECTED_PERSISTENCE_DIAGRAM = True` now draws a binned connected persistence diagram: dots and segment endpoints are aggregated into PD-coordinate bins, while connected segments are overlaid in a fixed contrasting color. The parameter-space summary is a discrete 2D binned histogram over `(radius, function value)`. The old translucent interval-support overlay remains available through `DRAW_PARAMETER_SPACE_INTERVALS = True`. By default this notebook draws all nonzero intervals. If `FILTER_PARAMETER_SPACE_INTERVALS_BY_ROW_LIFETIME = True`, it only draws the top `PARAMETER_SPACE_LIFETIME_FILTER_TOP_FRACTION` intervals by lifetime on row `PARAMETER_SPACE_LIFETIME_FILTER_ROW`; choose a row that intersects the intervals.


In [ ]:
import importlib
import intmult_from_filtration as _intmult_from_filtration
import interval_multiplicity_visualization as _interval_multiplicity_visualization

_intmult_from_filtration = importlib.reload(_intmult_from_filtration)
_interval_multiplicity_visualization = importlib.reload(_interval_multiplicity_visualization)
load_interval_multiplicities_text = _intmult_from_filtration.load_interval_multiplicities_text
interval_multiplicity_visualization_mode = _interval_multiplicity_visualization.interval_multiplicity_visualization_mode
load_grid_compression_from_scc2020 = _interval_multiplicity_visualization.load_grid_compression_from_scc2020
plot_connected_persistence_diagram_binned_histogram_for_two_row_filtration = _interval_multiplicity_visualization.plot_connected_persistence_diagram_binned_histogram_for_two_row_filtration
plot_connected_persistence_diagram_for_two_row_filtration = _interval_multiplicity_visualization.plot_connected_persistence_diagram_for_two_row_filtration
plot_interval_multiplicity_binned_2d_histogram = _interval_multiplicity_visualization.plot_interval_multiplicity_binned_2d_histogram
plot_interval_multiplicities_on_grid = _interval_multiplicity_visualization.plot_interval_multiplicities_on_grid
select_interval_indices_by_row_lifetime = _interval_multiplicity_visualization.select_interval_indices_by_row_lifetime
summarize_interval_multiplicity_result = _interval_multiplicity_visualization.summarize_interval_multiplicity_result

DRAW_CONNECTED_PERSISTENCE_DIAGRAM = False  # only works for 2-row compressed grids
DRAW_COVERAGE_2D_HISTOGRAM = False
DRAW_PARAMETER_SPACE_INTERVALS = True  # old translucent interval-support overlay
FILTER_PARAMETER_SPACE_INTERVALS_BY_ROW_LIFETIME = False  # False draws all nonzero intervals.
PARAMETER_SPACE_LIFETIME_FILTER_ROW = 0  # If filtering is enabled, choose a row intersecting the intervals.
PARAMETER_SPACE_LIFETIME_FILTER_TOP_FRACTION = 1  # If filtering is enabled, choose the top fraction of intervals by lifetime on the chosen row.
PARAMETER_SPACE_LIFETIME_FILTER_INCLUDE_TIES = True

VISUALIZATION_MULTIPLICITY_TXT = MULTIPLICITY_TXT  # switch to INJECTIVE_MULTIPLICITY_TXT if desired
CONNECTED_PD_HISTOGRAM_BINS = 60
CONNECTED_PD_HISTOGRAM_CMAP = "viridis"
CONNECTED_PD_HISTOGRAM_TICK_COUNT = 8  # set None for automatic sparse ticks
CONNECTED_PD_HISTOGRAM_SHOW_GRID = True
CONNECTED_PD_HISTOGRAM_GRID_COLOR = "black"
CONNECTED_PD_HISTOGRAM_GRID_LINEWIDTH = 0.25
CONNECTED_PD_HISTOGRAM_GRID_ALPHA = 0.65
CONNECTED_PD_SEGMENT_COLOR = "#00D5FF"
CONNECTED_PD_SEGMENT_LINEWIDTH = 0.5
CONNECTED_PD_SEGMENT_ENDPOINT_SIZE = 5
CONNECTED_PD_SEGMENT_ENDPOINT_MARKER = "o"  # e.g. "s", "o", "^", "D", "x"
COVERAGE_2D_HISTOGRAM_X_BINS = 30
COVERAGE_2D_HISTOGRAM_Y_BINS = None  # None keeps exact function rows; set an integer to bin y too
COVERAGE_2D_HISTOGRAM_AGGREGATION = "max"  # sum, mean, or max
COVERAGE_2D_HISTOGRAM_CMAP = "viridis"
INTERVAL_GRID_MAX_INTERVALS = None  # set an integer to show only the first N nonzero intervals
INTERVAL_GRID_SORT_BY = "index"  # index, multiplicity, or size
INTERVAL_GRID_ALPHA = 0.1
INTERVAL_GRID_LINEWIDTH = 0.3
INTERVAL_GRID_DRAW_INTERNAL_CELL_EDGES = False
INTERVAL_GRID_DRAW_OUTLINES = True
INTERVAL_GRID_OUTLINE_LINEWIDTH = 1.6
INTERVAL_GRID_OUTLINE_ALPHA = 1.0
INTERVAL_GRID_OUTLINE_COLOR = None  # None uses interval-index colors; set e.g. "black" for one fixed color
INTERVAL_GRID_OUTLINE_CMAP = "Set1"
INTERVAL_GRID_OUTLINE_HALO_COLOR = "black"  # set None to disable the contrast halo
INTERVAL_GRID_OUTLINE_HALO_LINEWIDTH = 6.0
INTERVAL_GRID_OUTLINE_HALO_ALPHA = 0.9
INTERVAL_GRID_DRAW_SHADOWS = True
INTERVAL_GRID_SHADOW_COLOR = "black"
INTERVAL_GRID_SHADOW_ALPHA = 0.12
INTERVAL_GRID_SHADOW_LINEWIDTH = None  # None uses a wider line than the outline halo
INTERVAL_GRID_USE_TRUE_RADIUS_SCALE = False  # True draws interval widths proportional to actual radius values
INTERVAL_GRID_AUTO_CROP_SELECTED_X = True
INTERVAL_GRID_X_CROP_PADDING = None  # None chooses automatic padding; use a number for manual padding
INTERVAL_GRID_X_CROP_LEFT_PADDING = 20  # overrides INTERVAL_GRID_X_CROP_PADDING on the left when set
INTERVAL_GRID_X_CROP_RIGHT_PADDING = 8  # overrides INTERVAL_GRID_X_CROP_PADDING on the right when set
INTERVAL_GRID_CLAMP_X_CROP_TO_DATA = False  # False keeps right-side padding even at the data boundary
INTERVAL_GRID_SHOW_SRC_SNK_X_TICKS = True
INTERVAL_GRID_PRESERVE_CELL_ASPECT = None  # None=auto; True=square cells; False=stretched y-axis
INTERVAL_GRID_COMPRESSION = (
    load_grid_compression_from_scc2020(CHAIN_COMPLEX_TXT)
    if CHAIN_COMPLEX_TXT.exists()
    else None
)

if "multiplicity_result" in globals():
    visualization_multiplicity_result = multiplicity_result
    print("Using in-memory multiplicity_result.")
elif VISUALIZATION_MULTIPLICITY_TXT.exists():
    visualization_multiplicity_result = load_interval_multiplicities_text(VISUALIZATION_MULTIPLICITY_TXT)
    print("Loaded saved multiplicity result:", VISUALIZATION_MULTIPLICITY_TXT)
else:
    raise FileNotFoundError(
        "No in-memory multiplicity_result and saved multiplicity txt was not found: "
        f"{VISUALIZATION_MULTIPLICITY_TXT}"
    )

interval_viz_summary = summarize_interval_multiplicity_result(visualization_multiplicity_result)
interval_viz_mode = interval_multiplicity_visualization_mode(visualization_multiplicity_result)
is_two_row_grid = interval_viz_mode == "connected_persistence_diagram"
print(interval_viz_summary)
print("Visualization mode:", interval_viz_mode)
if INTERVAL_GRID_COMPRESSION is None:
    print("Original scc2020 chain complex not found; showing compressed tick labels only.")

fig_connected_pd = ax_connected_pd = None
if DRAW_CONNECTED_PERSISTENCE_DIAGRAM and is_two_row_grid:
    fig_connected_pd, ax_connected_pd = plot_connected_persistence_diagram_binned_histogram_for_two_row_filtration(
        visualization_multiplicity_result,
        max_intervals=INTERVAL_GRID_MAX_INTERVALS,
        sort_by=INTERVAL_GRID_SORT_BY,
        compression=INTERVAL_GRID_COMPRESSION,
        bins=CONNECTED_PD_HISTOGRAM_BINS,
        cmap=CONNECTED_PD_HISTOGRAM_CMAP,
        tick_count=CONNECTED_PD_HISTOGRAM_TICK_COUNT,
        show_grid=CONNECTED_PD_HISTOGRAM_SHOW_GRID,
        grid_color=CONNECTED_PD_HISTOGRAM_GRID_COLOR,
        grid_linewidth=CONNECTED_PD_HISTOGRAM_GRID_LINEWIDTH,
        grid_alpha=CONNECTED_PD_HISTOGRAM_GRID_ALPHA,
        segment_color=CONNECTED_PD_SEGMENT_COLOR,
        segment_linewidth=CONNECTED_PD_SEGMENT_LINEWIDTH,
        segment_endpoint_size=CONNECTED_PD_SEGMENT_ENDPOINT_SIZE,
        segment_endpoint_marker=CONNECTED_PD_SEGMENT_ENDPOINT_MARKER,
        title=f"Binned connected persistence diagram: {OUTPUT_PREFIX}",
    )
elif DRAW_CONNECTED_PERSISTENCE_DIAGRAM:
    print("Connected persistence diagram skipped: compressed grid is not 2-row.")

fig_coverage_2d_histogram = ax_coverage_2d_histogram = None
if DRAW_COVERAGE_2D_HISTOGRAM:
    fig_coverage_2d_histogram, ax_coverage_2d_histogram = plot_interval_multiplicity_binned_2d_histogram(
        visualization_multiplicity_result,
        x_bins=COVERAGE_2D_HISTOGRAM_X_BINS,
        y_bins=COVERAGE_2D_HISTOGRAM_Y_BINS,
        aggregation=COVERAGE_2D_HISTOGRAM_AGGREGATION,
        cmap=COVERAGE_2D_HISTOGRAM_CMAP,
        compression=INTERVAL_GRID_COMPRESSION,
        title=f"2D binned coverage histogram of nonzero intervals: {OUTPUT_PREFIX}",
    )

fig_parameter_spaces = ax_parameter_spaces = None
parameter_space_interval_indices = None
if DRAW_PARAMETER_SPACE_INTERVALS and FILTER_PARAMETER_SPACE_INTERVALS_BY_ROW_LIFETIME:
    parameter_space_interval_indices = select_interval_indices_by_row_lifetime(
        visualization_multiplicity_result,
        row=PARAMETER_SPACE_LIFETIME_FILTER_ROW,
        top_fraction=PARAMETER_SPACE_LIFETIME_FILTER_TOP_FRACTION,
        include_ties=PARAMETER_SPACE_LIFETIME_FILTER_INCLUDE_TIES,
    )
    print(
        f"Parameter-space interval filter: selected {len(parameter_space_interval_indices)} "
        f"intervals in top {100 * PARAMETER_SPACE_LIFETIME_FILTER_TOP_FRACTION:.2g}% "
        f"by lifetime on y={PARAMETER_SPACE_LIFETIME_FILTER_ROW}."
    )
    if not parameter_space_interval_indices:
        parameter_space_interval_indices = None
        print("No intervals intersect the selected filter row; drawing all nonzero intervals instead.")

if DRAW_PARAMETER_SPACE_INTERVALS:
    fig_parameter_spaces, ax_parameter_spaces = plot_interval_multiplicities_on_grid(
        visualization_multiplicity_result,
        interval_indices=parameter_space_interval_indices,
        max_intervals=INTERVAL_GRID_MAX_INTERVALS if parameter_space_interval_indices is None else None,
        sort_by=INTERVAL_GRID_SORT_BY,
        alpha=INTERVAL_GRID_ALPHA,
        linewidth=INTERVAL_GRID_LINEWIDTH,
        draw_internal_cell_edges=INTERVAL_GRID_DRAW_INTERNAL_CELL_EDGES,
        draw_outlines=INTERVAL_GRID_DRAW_OUTLINES,
        outline_linewidth=INTERVAL_GRID_OUTLINE_LINEWIDTH,
        outline_alpha=INTERVAL_GRID_OUTLINE_ALPHA,
        outline_color=INTERVAL_GRID_OUTLINE_COLOR,
        outline_cmap=INTERVAL_GRID_OUTLINE_CMAP,
        outline_halo_color=INTERVAL_GRID_OUTLINE_HALO_COLOR,
        outline_halo_linewidth=INTERVAL_GRID_OUTLINE_HALO_LINEWIDTH,
        outline_halo_alpha=INTERVAL_GRID_OUTLINE_HALO_ALPHA,
        draw_shadows=INTERVAL_GRID_DRAW_SHADOWS,
        shadow_color=INTERVAL_GRID_SHADOW_COLOR,
        shadow_alpha=INTERVAL_GRID_SHADOW_ALPHA,
        shadow_linewidth=INTERVAL_GRID_SHADOW_LINEWIDTH,
        use_true_radius_scale=INTERVAL_GRID_USE_TRUE_RADIUS_SCALE,
        crop_x_to_intervals=(INTERVAL_GRID_AUTO_CROP_SELECTED_X and parameter_space_interval_indices is not None),
        x_crop_padding=INTERVAL_GRID_X_CROP_PADDING,
        x_crop_left_padding=INTERVAL_GRID_X_CROP_LEFT_PADDING,
        x_crop_right_padding=INTERVAL_GRID_X_CROP_RIGHT_PADDING,
        clamp_x_crop_to_data=INTERVAL_GRID_CLAMP_X_CROP_TO_DATA,
        show_src_snk_x_ticks=(INTERVAL_GRID_SHOW_SRC_SNK_X_TICKS and parameter_space_interval_indices is not None and INTERVAL_GRID_AUTO_CROP_SELECTED_X),
        only_src_snk_x_ticks=(FILTER_PARAMETER_SPACE_INTERVALS_BY_ROW_LIFETIME and parameter_space_interval_indices is not None),
        preserve_cell_aspect=INTERVAL_GRID_PRESERVE_CELL_ASPECT,
        compression=INTERVAL_GRID_COMPRESSION,
        title=(
            f"Filtered interval parameter spaces on compression grid: {OUTPUT_PREFIX}"
            if parameter_space_interval_indices is not None
            else f"All nonzero interval parameter spaces on compression grid: {OUTPUT_PREFIX}"
        ),
    )


<!-- ## Optional: Filter Externally Supplied Intervals

For large grids, automatic support-antichain generation may be too large. In that case, provide intervals manually or from another generator, then use the same conservative filter. -->

In [ ]:
# external_intervals = [
#     Interval([(0, 0)], [(1, 2), (2, 1)]),
# ]

# filtered_external = filter_interval_candidates(
#     external_intervals,
#     support_data,
#     criterion=CRITERION,
# )

# print("Retained external intervals:", len(filtered_external))
# for candidate in filtered_external:
#     print("src:", candidate.src, "snk:", candidate.snk)

## Notes

The candidate txt records the conservative Step 1 filter. The multiplicity txt records only intervals whose computed multiplicity is nonzero, together with the two ranks used in the formula.